In [9]:
from sklearn.datasets import make_classification
import numpy as np
import xgboost as xgb

seed = 1994

X, y = make_classification(random_state=seed)
rng = np.random.default_rng(seed)
n_query_groups = 3
qid = rng.integers(0, n_query_groups, size=X.shape[0])
#print(f"before sort: {X[:2]}, {y[:2]}, {qid[:2]}")

#sort based on query index
sorted_index = np.argsort(qid)
X = X[sorted_index, :]
y = y[sorted_index]
qid = qid[sorted_index]
#print(f"after sort: {X[:2]}, {y[:2]}, {qid[:2]}")

In [10]:
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold, cross_val_score


ranker = xgb.XGBRanker(tree_method="hist", 
                       lambdarank_num_pair_per_sample=8, 
                       objective="rank:ndcg", 
                       lambdarank_pair_method="topk")
ranker.fit(X, y, qid=qid)



XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [11]:
df=pd.DataFrame(X, columns = [str(i) for i in range(X.shape[1])])
df['qid'] = qid

ranker.fit(df, y)

kfold = StratifiedGroupKFold(shuffle=False)
cross_val_score(ranker, df, y, groups=df.qid, cv=kfold)

/opt/miniconda3/envs/housing/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:04:02] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1744329043786/work/src/common/error_msg.cc:52: Empty dataset at worker: 0
  warnings.warn(smsg, UserWarning)


array([1., 1., 1., 0., 0.])

In [13]:
scores = ranker.predict(X)
sorted_idx = np.argsort(scores)[::-1]
# Sort the relevance scores from most relevant to least relevant
scores = scores[sorted_idx]
print(scores)

[ 1.7109468   1.7109468   1.7109468   1.7109468   1.7109468   1.7109468
  1.7109468   1.7109468   1.7109468   1.7109468   1.7109468   1.7109468
  1.7109468   1.7109468   1.7109468   1.7109468   1.7109468   1.7109468
  1.7109468   1.7109468   1.7109468   1.7109468   1.5921135   1.5921135
  1.5921135   1.5420871   1.5420871   1.5420871   1.5420871   1.5420871
  1.4591678   1.4591678   1.4232538   1.4232538   1.3153265   1.3153265
  0.97181636  0.4532021  -0.16912135 -0.4710336  -0.4710336  -1.0312457
 -1.0362681  -1.0362681  -1.2228703  -1.2240134  -1.2240134  -1.2240134
 -1.2240134  -1.2240134  -1.2240134  -1.2240134  -1.3428468  -1.3428468
 -1.3428468  -1.4160769  -1.4160769  -1.4160769  -1.4757924  -1.4757924
 -1.4757924  -1.6504426  -1.6504426  -1.6504426  -1.6504426  -1.6504426
 -1.655904   -1.655904   -1.655904   -1.655904   -1.655904   -1.655904
 -1.655904   -1.655904   -1.655904   -1.655904   -1.655904   -1.655904
 -1.655904   -1.655904   -1.655904   -1.655904   -1.8436491  -1.84

In [15]:
from sklearn.metrics import ndcg_score
import numpy as np

# true relevance scores (e.g. 1 if clicked, 0 if not)
true_relevance = np.array([[1, 0, 1, 0, 0]])

# your model's predicted scores
predicted_scores = np.array([[0.9, 0.2, 0.7, 0.1, 0.4]])

score = ndcg_score(true_relevance, predicted_scores)
print(f"NDCG score: {score}")

NDCG score: 1.0
